# Watch a perceptron learn

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/data-analysis/perceptron-learning-animation.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/data-analysis/perceptron-learning-animation.ipynb)

How does a classifier move its decision boundary after making a mistake? Start with flower measurements, calculate one update by hand, and watch the same rule learn from data.

This is the curated edition of the **ABW Lecture 4 perceptron animation**, supplied in the [original lecture notebook](https://colab.research.google.com/drive/1O49Lo9_Y_-DUydYKwh4jVRXt214zoDky). It retains Iris exploration, selectable feature/species pairs, the perceptron algorithm, the animated boundary, mistake highlighting, weights, error counts and GIF export.

**By the end:** explain the bias and weights; trace an update; read an animation; distinguish updates from epochs and training errors from test performance; explain a case where a straight boundary cannot work.

Allow about 30–45 minutes. Basic Python loops, functions and arrays are useful. Run all cells in order, then change one setting at a time. The animation controls appear after execution in Colab, Binder or a trusted Jupyter notebook; GitHub's static preview does not play them.

## 1. Prepare the tools

NumPy handles vectors, pandas holds the measurements, and Matplotlib draws the animation. The shared teaching helper installs only missing packages. The learning and drawing functions are displayed in full below because their behavior is the subject of this lesson.

In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "PIL": "pillow"}
ensure_packages(required_packages)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import ListedColormap
from IPython.display import HTML, Image, display

## 2. Look at the flowers before fitting a line

The original UCI table has 150 flowers, four measurements in centimetres and three species. We keep the original `iris.data` bytes used in the lecture, including the two historical discrepancies documented by UCI, so this lesson does not silently substitute a different Iris edition. A checked-in copy makes repeated classroom runs reproducible.

Data credit: Fisher, R. (1936), *Iris*, UCI Machine Learning Repository, [DOI and dataset documentation](https://doi.org/10.24432/C56C76), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
# Prefer the repository copy; Colab fetches the same checksum-verified file.
data_path = next((folder / "data/iris.data" for folder in [Path.cwd(), *Path.cwd().parents]
                  if (folder / "data/iris.data").is_file()), None)
if data_path is None:
    data_path = Path("iris.data")
    content = urlopen("https://raw.githubusercontent.com/gromicho/teaching/main/data/iris.data", timeout=45).read()
    data_path.write_bytes(content)
expected = "6f608b71a7317216319b4d27b4d9bc84e6abd734eda7872b71a458569e2656c0"
if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected:
    raise ValueError("The Iris file differs from the recorded lecture dataset.")

columns = ["Sepal length", "Sepal width", "Petal length", "Petal width", "Species"]
iris = pd.read_csv(data_path, header=None, names=columns)
assert iris.shape == (150, 5) and not iris.isna().any().any()
display(iris.head())
display(iris.groupby("Species").size().rename("flowers"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, features in zip(axes, [("Sepal length", "Sepal width"), ("Petal length", "Petal width")]):
    for species, group in iris.groupby("Species"):
        ax.scatter(group[features[0]], group[features[1]], label=species.removeprefix("Iris-"), s=25)
    ax.set(xlabel=f"{features[0]} (cm)", ylabel=f"{features[1]} (cm)")
axes[0].legend()
plt.show()

**Predict:** which two species look easiest to separate with one straight line? Would you choose petals or sepals?

The original experiment uses setosa versus versicolor and petal measurements. Select exactly two features and two species below. The first species gets label +1; the second gets −1. We keep the original row order by default.

In [ ]:
FEATURES = ["Petal length", "Petal width"]
SPECIES = ["Iris-setosa", "Iris-versicolor"]
LEARNING_RATE = 0.1
MAX_EPOCHS = 1000
SHUFFLE = False
SEED = 7

if len(FEATURES) != 2 or len(set(FEATURES)) != 2 or not set(FEATURES) <= set(columns[:4]):
    raise ValueError("Choose two distinct measurement columns.")
if len(SPECIES) != 2 or len(set(SPECIES)) != 2 or not set(SPECIES) <= set(iris.Species):
    raise ValueError("Choose two distinct Iris species.")
selected = iris.loc[iris.Species.isin(SPECIES)]
X = selected[FEATURES].to_numpy(dtype=float)
y = np.where(selected.Species.to_numpy() == SPECIES[0], 1, -1)
class_names = {1: SPECIES[0].removeprefix("Iris-"), -1: SPECIES[1].removeprefix("Iris-")}
display(selected.head())
print(f"{len(y)} flowers; label counts: +1={(y == 1).sum()}, -1={(y == -1).sum()}")

## 3. From a score to a prediction

For two measurements $x_1,x_2$, the score is
$$s=b+w_1x_1+w_2x_2.$$
Predict **+1 when $s\geq0$**, and **−1 otherwise**. The bias $b$ lets the boundary move away from the origin. The equation $s=0$ describes the boundary when at least one feature weight is nonzero. With all weights zero there is no unique line: our tie convention predicts +1 everywhere.

On a mistake, update
$$b\leftarrow b+\eta y,\qquad \mathbf{w}\leftarrow\mathbf{w}+\eta y\mathbf{x}.$$
Here $\eta>0$ is the learning rate. A correct prediction leaves the weights unchanged. We use an augmented vector $[1,x_1,x_2]$ to update the bias and feature weights together.

**By hand:** start at zero weights and see a point $[2,1]$ with label −1 and $\eta=0.1$. What is its initial prediction? What are the three updated weights? Does it change its prediction?

In [ ]:
def predict(X, weights):
    """The same boundary convention is used in training, scores and plots."""
    scores = weights[0] + np.asarray(X) @ weights[1:]
    return np.where(scores >= 0, 1, -1)

In [ ]:
point = np.array([2.0, 1.0])
label = -1
before = np.zeros(3)
after = before + 0.1 * label * np.r_[1.0, point]
print("Before:", before, "prediction:", predict(point, before))
print("After: ", after, "prediction:", predict(point, after))
assert np.allclose(after, [-0.1, -0.2, -0.1])
assert predict(point, after) == label

## 4. Record the learning process

An **update** is one weight change following a mistake. An **epoch** is one pass through all training observations. We record the initial weights and a separate copy after every update; otherwise later changes could overwrite our history.

At each epoch's end, count errors using the final weights on the whole training set. This differs from counting mistakes encountered during the pass, when the weights were still changing. Stop when the final model makes zero training errors, or when the epoch limit is reached. Reaching the limit is reported as non-convergence, not success.

In [ ]:
def train_perceptron(X, y, learning_rate=0.1, max_epochs=100, shuffle=False, seed=7):
    """Record the initial model and every actual mistake-driven update."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError("X must be a nonempty matrix.")
    if y.shape != (len(X),) or not np.isin(y, [-1, 1]).all():
        raise ValueError("Use one label, -1 or +1, per observation.")
    if not np.isfinite(X).all() or not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError("Use finite features and a positive finite learning rate.")
    if not isinstance(max_epochs, (int, np.integer)) or max_epochs < 1:
        raise ValueError("max_epochs must be a positive integer.")
    rng = np.random.default_rng(seed)
    weights = np.zeros(X.shape[1] + 1)
    history = [{"weights": weights.copy(), "epoch": 0, "sample": None, "update": 0}]
    epoch_rows = []
    for epoch in range(1, max_epochs + 1):
        mistakes = 0
        order = rng.permutation(len(X)) if shuffle else np.arange(len(X))
        for index in order:
            if predict(X[index], weights) != y[index]:
                augmented = np.r_[1.0, X[index]]
                weights += learning_rate * y[index] * augmented
                if not np.isfinite(weights).all():
                    raise ValueError("Weights overflowed; rescale the features or learning rate.")
                mistakes += 1
                history.append({"weights": weights.copy(), "epoch": epoch,
                                "sample": int(index), "update": len(history)})
        errors = int(np.count_nonzero(predict(X, weights) != y))
        epoch_rows.append({"epoch": epoch, "updates_this_epoch": mistakes,
                           "errors_at_epoch_end": errors})
        if errors == 0:
            break
    return {"weights": weights.copy(), "history": history, "epochs": epoch_rows,
            "converged": errors == 0}

In [ ]:
result = train_perceptron(X, y, learning_rate=LEARNING_RATE, max_epochs=MAX_EPOCHS,
                          shuffle=SHUFFLE, seed=SEED)
epoch_table = pd.DataFrame(result["epochs"])
display(epoch_table)
print("Zero training errors reached:", result["converged"])
print("Total weight updates:", len(result["history"]) - 1)
print("Final weights [bias, w1, w2]:", result["weights"])
assert np.array_equal(result["history"][-1]["weights"], result["weights"])

## 5. Watch the boundary move

Blue circles and orange triangles show the true classes. The background shows the predicted class using those same colors. Red rings mark observations currently misclassified. At zero weights the whole background predicts +1; a line appears once the model has a nonzero feature weight.

Use **play, pause and the frame slider**. Watch for an update that corrects one flower but makes another wrong. The title reports the actual epoch, update number and current training errors.

To keep playback small, long histories are sampled to at most 60 frames, always including the initial and final models. The training history is complete; the animation does not interpolate invented models. Increase `max_frames` below to see more recorded updates.

The following drawing code evaluates the prediction rule on a grid. This keeps shading and mistake counts consistent even when the boundary is vertical or the classifier is constant.

In [ ]:
def boundary_segment(weights, xlim, ylim):
    """Coordinates for a line; empty arrays mean a constant classifier."""
    bias, w1, w2 = weights
    if w2 != 0:
        xs = np.asarray(xlim)
        return xs, -(bias + w1 * xs) / w2
    if w1 != 0:
        return np.full(2, -bias / w1), np.asarray(ylim)
    return np.array([]), np.array([])

In [ ]:
def make_learning_animation(X, y, result, feature_names, class_names,
                            max_frames=60, interval=500):
    """Animate recorded models, sampling frames without changing training."""
    if X.shape[1] != 2 or len(feature_names) != 2:
        raise ValueError("The animation requires exactly two features.")
    if max_frames < 2:
        raise ValueError("Keep at least the initial and final frames.")
    history = result["history"]
    frame_indices = np.unique(np.linspace(0, len(history) - 1,
                                         min(max_frames, len(history)), dtype=int))
    fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
    xlim = (X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
    ylim = (X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
    gx, gy = np.meshgrid(np.linspace(*xlim, 120), np.linspace(*ylim, 120))
    grid = np.column_stack([gx.ravel(), gy.ravel()])
    colors = {1: "#2878B5", -1: "#D68124"}
    region = ax.imshow(np.zeros_like(gx), origin="lower", extent=(*xlim, *ylim),
                       cmap=ListedColormap([colors[-1], colors[1]]), vmin=-1, vmax=1,
                       alpha=0.16, aspect="auto", interpolation="nearest")
    for label, marker in [(1, "o"), (-1, "^")]:
        points = X[y == label]
        ax.scatter(*points.T, c=colors[label], marker=marker, s=55,
                   edgecolors="white", linewidths=0.5,
                   label=f"{class_names[label]} ({label:+d})")
    line, = ax.plot([], [], color="#263238", lw=2, label="Decision boundary")
    errors_artist = ax.scatter([], [], facecolors="none", edgecolors="crimson",
                               s=130, linewidths=1.5, label="Currently misclassified")
    ax.set(xlim=xlim, ylim=ylim, xlabel=feature_names[0], ylabel=feature_names[1])
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(alpha=0.15)

    def draw_frame(frame):
        snapshot = history[frame_indices[frame]]
        weights = snapshot["weights"]
        predictions = predict(X, weights)
        wrong = predictions != y
        region.set_data(predict(grid, weights).reshape(gx.shape))
        line.set_data(*boundary_segment(weights, xlim, ylim))
        errors_artist.set_offsets(X[wrong])
        ax.set_title(
            f"Epoch {snapshot['epoch']} | Update {snapshot['update']} | "
            f"Training errors {wrong.sum()}/{len(y)}\n"
            f"b={weights[0]:.3f}, w1={weights[1]:.3f}, w2={weights[2]:.3f}"
        )
        return region, line, errors_artist

    ani = FuncAnimation(fig, draw_frame, frames=len(frame_indices), interval=interval,
                        repeat=False, blit=False)
    return ani, fig, draw_frame, frame_indices

In [ ]:
animation, animation_figure, draw_frame, shown_frames = make_learning_animation(
    X, y, result, [f"{name} (cm)" for name in FEATURES], class_names,
    max_frames=60, interval=500,
)
print(f"Displaying {len(shown_frames)} of {len(result['history'])} recorded models.")
animation_html = animation.to_jshtml(fps=2, default_mode="once")
plt.close(animation_figure)
display(HTML(animation_html))

## 6. Inspect what the animation means

The error count need not decrease after each update. Correcting the current point does not guarantee improvement on every other point. Also, a fitted line is not necessarily the widest-margin line.

The table above and the plot below measure **training performance**. They do not estimate accuracy on new flowers. For that, hold out test observations before choosing features or tuning settings, and evaluate the resulting fixed model on them.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
ax.plot(epoch_table.epoch, epoch_table.errors_at_epoch_end, marker="o", label="Errors at epoch end")
ax.plot(epoch_table.epoch, epoch_table.updates_this_epoch, marker=".", label="Updates during epoch")
ax.set(xlabel="Epoch", ylabel="Count", title="Two different ways to describe learning")
ax.legend()
plt.show()

## 7. Change one thing at a time

1. Change `SHUFFLE` to `True`. Does the same dataset yield the same final line? Keep `SEED` fixed to reproduce a run.
2. Keep zero initialization and the observation order fixed, and try learning rates 0.01, 0.1 and 1. In exact arithmetic a positive constant learning rate scales all weights together here, preserving predictions. Floating-point ties can differ. Does a larger value actually mean fewer updates?
3. Compare the sepal and petal feature pairs. Then compare versicolor with virginica. Do not interpret reaching the epoch limit as proof of non-separability.
4. After editing the settings cell, rerun from there to update training and the animation together.

Here is a reproducible comparison using a bounded epoch budget. It reports actual training runs, including ones that fail to reach zero errors.

In [ ]:
comparisons = []
for pair, features, shuffled in [
    (["Iris-setosa", "Iris-versicolor"], ["Petal length", "Petal width"], False),
    (["Iris-setosa", "Iris-versicolor"], ["Petal length", "Petal width"], True),
    (["Iris-setosa", "Iris-versicolor"], ["Sepal length", "Sepal width"], False),
    (["Iris-versicolor", "Iris-virginica"], ["Petal length", "Petal width"], False),
]:
    subset = iris.loc[iris.Species.isin(pair)]
    trial_X = subset[features].to_numpy(dtype=float)
    trial_y = np.where(subset.Species.to_numpy() == pair[0], 1, -1)
    trial = train_perceptron(trial_X, trial_y, max_epochs=100, shuffle=shuffled, seed=7)
    comparisons.append({"species": " / ".join(pair), "features": " / ".join(features),
                        "shuffle": shuffled, "epochs": len(trial["epochs"]),
                        "updates": len(trial["history"]) - 1,
                        "final training errors": trial["epochs"][-1]["errors_at_epoch_end"],
                        "zero errors reached": trial["converged"]})
display(pd.DataFrame(comparisons))

## 8. A line cannot solve every problem

The perceptron convergence guarantee assumes linearly separable data and the usual finite-data, positive-step setting. Consider XOR: the two off-diagonal corners are +1 and the other two corners are −1.

**Draw it first:** can one straight line put both +1 corners on its positive side and both −1 corners on its negative side?

For the two +1 points we would need `b + w1 >= 0` and `b + w2 >= 0`. For the two −1 points we would need `b < 0` and `b + w1 + w2 < 0`. Adding each pair demands that the same expression is both nonnegative and negative. A perfect linear classifier is impossible.

In [ ]:
xor_X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
xor_y = np.array([-1, 1, 1, -1])
xor_result = train_perceptron(xor_X, xor_y, max_epochs=12)
assert not xor_result["converged"]
print("Epoch limit reached; final errors:", xor_result["epochs"][-1]["errors_at_epoch_end"])
xor_animation, xor_figure, _, _ = make_learning_animation(
    xor_X, xor_y, xor_result, ["x1", "x2"], {1: "different inputs", -1: "equal inputs"},
    max_frames=20, interval=500,
)
xor_html = xor_animation.to_jshtml(fps=2, default_mode="once")
plt.close(xor_figure)
display(HTML(xor_html))

## 9. Optional: export the animation for lecture slides

The original notebook saved an animated GIF. Enable the switch below to save the Iris animation with Pillow; no external video encoder is required. The path is relative to the current notebook session, so it works outside Colab too. In Colab, download the generated file from the Files panel. A GIF contains the selected display frames, while the complete learning history remains in `result`.

In [ ]:
SAVE_GIF = False
if SAVE_GIF:
    gif_path = Path("perceptron-learning.gif")
    animation.save(gif_path, writer=PillowWriter(fps=2), dpi=80)
    display(Image(filename=str(gif_path)))
    print("Saved:", gif_path.resolve())

## Explain it without the code

- What do the bias and the two weights control?
- Why can an update introduce a new mistake elsewhere?
- Why are mistakes during an epoch different from errors at its end?
- What does zero training error tell us, and what does it leave unanswered?
- Why does increasing the epoch limit fail to solve XOR?

For a next experiment, add a nonlinear feature to XOR and explain how the model can be linear in its features while having a nonlinear boundary in the original coordinates.

### Sources and curation

The original ABW Lecture 4 notebook supplied the Iris exploration and perceptron animation. This edition adds the mathematical narrative, records individual updates, corrects inconsistent zero-score predictions and boundary shading, handles vertical and constant classifiers, and adds reproducible comparisons and XOR. The algorithm and animation implementations remain visible for study.

Dataset: [UCI Iris](https://archive.ics.uci.edu/dataset/53/iris). Animation API: [Matplotlib animation and HTML/GIF export](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.Animation.html).